# 📱 Android Emulator in Google Colab with GPU

## 🚀 Run Android APKs on Real Phone Screen Sizes

This notebook sets up a full Android emulator with:
- ✅ GPU acceleration (if available)
- ✅ Multiple phone screen sizes (Pixel, Galaxy, etc.)
- ✅ Remote desktop access via web browser
- ✅ APK installation and testing
- ✅ ADB access for debugging

---

### 📋 Prerequisites:
1. Google account
2. Enable GPU in Colab: **Runtime → Change runtime type → T4 GPU**
3. Your APK file ready to upload

### ⏱️ Estimated Setup Time: 5-10 minutes

---

## Step 1: Check GPU and System Info

In [ ]:
# Check if GPU is available
!nvidia-smi
print("\n" + "="*50)
print("System Information:")
print("="*50)
!cat /proc/cpuinfo | grep "model name" | head -1
!free -h | grep "Mem:"
!df -h | grep "/dev/root"

## Step 2: Install Required Dependencies

In [ ]:
%%bash

# Update system
echo "📦 Updating system packages..."
apt-get update -qq > /dev/null 2>&1

# Install essential tools
echo "🔧 Installing essential tools..."
apt-get install -y -qq \
    wget \
    unzip \
    openjdk-11-jdk \
    xvfb \
    x11vnc \
    fluxbox \
    wget \
    wmctrl \
    xdotool \
    > /dev/null 2>&1

# Install noVNC for web-based VNC
echo "🌐 Installing noVNC for browser access..."
cd /content
git clone -q https://github.com/novnc/noVNC.git > /dev/null 2>&1
git clone -q https://github.com/novnc/websockify /content/noVNC/utils/websockify > /dev/null 2>&1

echo "✅ Dependencies installed successfully!"

## Step 3: Install Android SDK and Tools

In [ ]:
%%bash

cd /content

# Download Android Command Line Tools
echo "📥 Downloading Android SDK Command Line Tools..."
wget -q https://dl.google.com/android/repository/commandlinetools-linux-9477386_latest.zip

# Extract
echo "📂 Extracting Android SDK..."
mkdir -p /content/android-sdk/cmdline-tools
unzip -q commandlinetools-linux-9477386_latest.zip -d /content/android-sdk/cmdline-tools
mv /content/android-sdk/cmdline-tools/cmdline-tools /content/android-sdk/cmdline-tools/latest

# Set environment variables
export ANDROID_HOME=/content/android-sdk
export PATH=$PATH:$ANDROID_HOME/cmdline-tools/latest/bin:$ANDROID_HOME/platform-tools:$ANDROID_HOME/emulator

# Save env vars
echo "export ANDROID_HOME=/content/android-sdk" >> ~/.bashrc
echo "export PATH=\$PATH:\$ANDROID_HOME/cmdline-tools/latest/bin:\$ANDROID_HOME/platform-tools:\$ANDROID_HOME/emulator" >> ~/.bashrc

echo "✅ Android SDK installed!"

## Step 4: Accept Licenses and Install Platform Tools

In [ ]:
import os
os.environ['ANDROID_HOME'] = '/content/android-sdk'
os.environ['PATH'] = f"{os.environ['PATH']}:/content/android-sdk/cmdline-tools/latest/bin:/content/android-sdk/platform-tools:/content/android-sdk/emulator"

# Accept licenses
!yes | /content/android-sdk/cmdline-tools/latest/bin/sdkmanager --licenses > /dev/null 2>&1

print("📦 Installing platform-tools, emulator, and system images...")
print("⏳ This may take 3-5 minutes...\n")

# Install required packages
!/content/android-sdk/cmdline-tools/latest/bin/sdkmanager \
    "platform-tools" \
    "platforms;android-33" \
    "emulator" \
    "system-images;android-33;google_apis;x86_64"

print("\n✅ Android tools installed successfully!")

## Step 5: Choose Your Phone Model

Select from popular Android phone configurations:

In [ ]:
# Phone configurations dictionary
PHONE_CONFIGS = {
    "1": {
        "name": "Google Pixel 7 Pro",
        "device": "pixel_7_pro",
        "skin": "pixel_7_pro",
        "width": 1440,
        "height": 3120,
        "dpi": 560
    },
    "2": {
        "name": "Samsung Galaxy S23",
        "device": "pixel_6",  # Similar specs
        "skin": "pixel_6",
        "width": 1080,
        "height": 2340,
        "dpi": 425
    },
    "3": {
        "name": "Google Pixel 6",
        "device": "pixel_6",
        "skin": "pixel_6",
        "width": 1080,
        "height": 2400,
        "dpi": 411
    },
    "4": {
        "name": "OnePlus 11 (Budget)",
        "device": "pixel_5",
        "skin": "pixel_5",
        "width": 1080,
        "height": 2400,
        "dpi": 440
    },
    "5": {
        "name": "Small Phone (Budget)",
        "device": "pixel_4",
        "skin": "pixel_4",
        "width": 1080,
        "height": 2280,
        "dpi": 440
    },
    "6": {
        "name": "Custom Size",
        "device": "pixel_6",
        "skin": "pixel_6",
        "width": None,  # Will be asked
        "height": None,
        "dpi": None
    }
}

# Display options
print("📱 Available Phone Models:\n")
print("="*60)
for key, config in PHONE_CONFIGS.items():
    if config['width']:
        print(f"{key}. {config['name']}")
        print(f"   Resolution: {config['width']}x{config['height']} @ {config['dpi']} DPI")
    else:
        print(f"{key}. {config['name']} (Enter your own dimensions)")
    print()
print("="*60)

# Get user choice
choice = input("\n👉 Select phone model (1-6): ").strip()

if choice in PHONE_CONFIGS:
    selected_phone = PHONE_CONFIGS[choice]
    
    # If custom, ask for dimensions
    if choice == "6":
        selected_phone['width'] = int(input("Enter width (e.g., 1080): "))
        selected_phone['height'] = int(input("Enter height (e.g., 2400): "))
        selected_phone['dpi'] = int(input("Enter DPI (e.g., 420): "))
    
    print(f"\n✅ Selected: {selected_phone['name']}")
    print(f"📐 Resolution: {selected_phone['width']}x{selected_phone['height']} @ {selected_phone['dpi']} DPI")
else:
    print("❌ Invalid choice! Using default: Pixel 6")
    selected_phone = PHONE_CONFIGS["3"]

# Save selection for next step
%store selected_phone

## Step 6: Create Android Virtual Device (AVD)

In [ ]:
# Retrieve selected phone config
%store -r selected_phone

import os
os.environ['ANDROID_HOME'] = '/content/android-sdk'
os.environ['PATH'] = f"{os.environ['PATH']}:/content/android-sdk/cmdline-tools/latest/bin:/content/android-sdk/platform-tools:/content/android-sdk/emulator"

# Create AVD
avd_name = "MyTestPhone"

print(f"🏗️ Creating Android Virtual Device: {selected_phone['name']}...\n")

# Create AVD with custom configuration
!/content/android-sdk/cmdline-tools/latest/bin/avdmanager create avd \
    --force \
    --name {avd_name} \
    --package "system-images;android-33;google_apis;x86_64" \
    --device "{selected_phone['device']}"

# Modify config.ini with custom resolution
config_path = f"/root/.android/avd/{avd_name}.avd/config.ini"

# Read existing config
with open(config_path, 'r') as f:
    config_lines = f.readlines()

# Modify resolution settings
new_config = []
for line in config_lines:
    if line.startswith('hw.lcd.width'):
        new_config.append(f"hw.lcd.width={selected_phone['width']}\n")
    elif line.startswith('hw.lcd.height'):
        new_config.append(f"hw.lcd.height={selected_phone['height']}\n")
    elif line.startswith('hw.lcd.density'):
        new_config.append(f"hw.lcd.density={selected_phone['dpi']}\n")
    else:
        new_config.append(line)

# Add GPU acceleration
new_config.append("hw.gpu.enabled=yes\n")
new_config.append("hw.gpu.mode=auto\n")

# Write back
with open(config_path, 'w') as f:
    f.writelines(new_config)

print(f"\n✅ AVD '{avd_name}' created successfully!")
print(f"📱 Device: {selected_phone['name']}")
print(f"📐 Resolution: {selected_phone['width']}x{selected_phone['height']}")
print(f"🎨 DPI: {selected_phone['dpi']}")

## Step 7: Start Virtual Display (Xvfb)

In [ ]:
import subprocess
import time

print("🖥️ Starting virtual display...\n")

# Start Xvfb (X virtual framebuffer)
xvfb_process = subprocess.Popen(
    ['Xvfb', ':99', '-screen', '0', '1920x1080x24'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Set DISPLAY environment variable
import os
os.environ['DISPLAY'] = ':99'

time.sleep(2)

# Start window manager
fluxbox_process = subprocess.Popen(
    ['fluxbox'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(2)

print("✅ Virtual display started on :99")

## Step 8: Start VNC Server for Remote Access

In [ ]:
import subprocess
import time

print("🌐 Starting VNC server...\n")

# Start x11vnc
vnc_process = subprocess.Popen(
    ['x11vnc', '-display', ':99', '-nopw', '-forever', '-xkb', '-shared'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(3)

print("✅ VNC server started on port 5900")

## Step 9: Start noVNC Web Server

In [ ]:
import subprocess
from google.colab import output
import time

print("🚀 Starting noVNC web server...\n")

# Start noVNC
novnc_process = subprocess.Popen(
    ['/content/noVNC/utils/novnc_proxy', '--vnc', 'localhost:5900', '--listen', '6080'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(3)

print("✅ noVNC started on port 6080\n")
print("="*60)
print("⚠️  Important: We need to expose the port using ngrok...")
print("   This will be done in the next step!")
print("="*60)

## Step 10: Expose VNC via Ngrok (Public URL)

In [ ]:
# Install pyngrok
!pip install -q pyngrok

from pyngrok import ngrok, conf
import time

# Optional: Set your ngrok auth token for better limits
# Get free token from: https://dashboard.ngrok.com/signup
# Uncomment and add your token:
# ngrok.set_auth_token("YOUR_NGROK_TOKEN_HERE")

print("🔗 Creating public URL with ngrok...\n")

# Open tunnel
public_url = ngrok.connect(6080, bind_tls=True)

print("\n" + "="*60)
print("🎉 SUCCESS! Your Android Emulator is accessible at:")
print("="*60)
print(f"\n🌐 VNC URL: {public_url}")
print(f"\n📱 Click the link above to access your Android phone!")
print("\n" + "="*60)
print("\n💡 Tip: Bookmark this URL - it's valid for this session")
print("\n⚠️  The emulator will start in the next step...")

## Step 11: Start Android Emulator

In [ ]:
import subprocess
import os
import time

os.environ['ANDROID_HOME'] = '/content/android-sdk'
os.environ['DISPLAY'] = ':99'
os.environ['PATH'] = f"{os.environ['PATH']}:/content/android-sdk/emulator:/content/android-sdk/platform-tools"

print("🚀 Starting Android Emulator...")
print("⏳ This will take 2-3 minutes to boot...\n")

# Start emulator in background
emulator_cmd = [
    '/content/android-sdk/emulator/emulator',
    '-avd', 'MyTestPhone',
    '-no-audio',
    '-no-boot-anim',
    '-gpu', 'swiftshader_indirect',  # Software GPU for Colab
    '-writable-system',
    '-no-snapshot-load'
]

emulator_process = subprocess.Popen(
    emulator_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("📱 Emulator is booting...")
print("\n🔍 Waiting for device to come online...\n")

# Wait for emulator to start
for i in range(60):
    result = subprocess.run(
        ['/content/android-sdk/platform-tools/adb', 'devices'],
        capture_output=True,
        text=True
    )
    
    if 'emulator-5554' in result.stdout:
        print("✅ Emulator is online!")
        break
    
    time.sleep(3)
    if i % 5 == 0:
        print(f"⏳ Still booting... ({i*3}s elapsed)")

# Wait for boot to complete
print("\n⏳ Waiting for Android to finish booting...\n")
time.sleep(20)

# Check boot status
!/content/android-sdk/platform-tools/adb wait-for-device
!/content/android-sdk/platform-tools/adb shell "while [ \"$(getprop sys.boot_completed)\" != \"1\" ]; do sleep 1; done"

print("\n" + "="*60)
print("🎉 ANDROID EMULATOR IS READY!")
print("="*60)
print("\n📱 Go to the VNC URL from Step 10 to see your phone!")
print("\n💡 Next: Upload and install your APK in the next step")

## Step 12: Upload and Install Your APK

In [ ]:
from google.colab import files
import os

print("📤 Upload your APK file:\n")

# Upload APK
uploaded = files.upload()

# Get uploaded filename
apk_file = list(uploaded.keys())[0]

print(f"\n📦 Installing {apk_file}...\n")

# Install APK
!/content/android-sdk/platform-tools/adb install -r "{apk_file}"

print("\n✅ APK installed successfully!")
print("\n📱 Go to the VNC browser window to see your app!")
print("   You can launch it from the app drawer")

## Step 13: Useful ADB Commands

In [ ]:
import os
os.environ['ANDROID_HOME'] = '/content/android-sdk'

print("📱 Android Device Info:\n")

# List connected devices
print("🔗 Connected Devices:")
!/content/android-sdk/platform-tools/adb devices

print("\n" + "="*60)

# Get device info
print("\n📊 Device Information:")
print("Android Version:")
!/content/android-sdk/platform-tools/adb shell getprop ro.build.version.release

print("\nSDK Version:")
!/content/android-sdk/platform-tools/adb shell getprop ro.build.version.sdk

print("\nDevice Model:")
!/content/android-sdk/platform-tools/adb shell getprop ro.product.model

print("\nScreen Resolution:")
!/content/android-sdk/platform-tools/adb shell wm size

print("\nScreen Density:")
!/content/android-sdk/platform-tools/adb shell wm density

print("\n" + "="*60)
print("\n📦 Installed Packages (3rd party):")
!/content/android-sdk/platform-tools/adb shell pm list packages -3

## Step 14: Additional Useful Commands

Run these commands as needed:

In [ ]:
# Choose an action
print("🛠️ Useful Commands:\n")
print("1. Take Screenshot")
print("2. Record Screen (10 seconds)")
print("3. View Logcat (logs)")
print("4. Uninstall APK")
print("5. Push file to device")
print("6. Pull file from device")
print("7. Restart ADB server")
print("8. Open App (enter package name)")

choice = input("\nSelect option (1-8): ").strip()

if choice == "1":
    # Screenshot
    !/content/android-sdk/platform-tools/adb shell screencap -p /sdcard/screenshot.png
    !/content/android-sdk/platform-tools/adb pull /sdcard/screenshot.png .
    print("✅ Screenshot saved as screenshot.png")
    from google.colab import files
    files.download('screenshot.png')

elif choice == "2":
    # Screen recording
    print("🎥 Recording for 10 seconds...")
    !/content/android-sdk/platform-tools/adb shell screenrecord --time-limit 10 /sdcard/recording.mp4
    !/content/android-sdk/platform-tools/adb pull /sdcard/recording.mp4 .
    print("✅ Recording saved as recording.mp4")
    from google.colab import files
    files.download('recording.mp4')

elif choice == "3":
    # Logcat
    print("📋 Last 50 log entries:\n")
    !/content/android-sdk/platform-tools/adb logcat -d -t 50

elif choice == "4":
    # Uninstall
    package = input("Enter package name (e.g., com.example.app): ")
    !/content/android-sdk/platform-tools/adb uninstall {package}
    print(f"✅ {package} uninstalled")

elif choice == "5":
    # Push file
    local_path = input("Local file path: ")
    remote_path = input("Remote path (e.g., /sdcard/file.txt): ")
    !/content/android-sdk/platform-tools/adb push {local_path} {remote_path}

elif choice == "6":
    # Pull file
    remote_path = input("Remote file path: ")
    !/content/android-sdk/platform-tools/adb pull {remote_path} .
    print("✅ File pulled to current directory")

elif choice == "7":
    # Restart ADB
    !/content/android-sdk/platform-tools/adb kill-server
    !/content/android-sdk/platform-tools/adb start-server
    print("✅ ADB server restarted")

elif choice == "8":
    # Launch app
    package = input("Enter package name: ")
    !/content/android-sdk/platform-tools/adb shell monkey -p {package} -c android.intent.category.LAUNCHER 1
    print(f"✅ Launched {package}")

else:
    print("❌ Invalid choice")

## 🎯 Quick Reference

### Essential Commands:

```bash
# Install APK
!adb install -r your-app.apk

# Uninstall APK
!adb uninstall com.your.package

# List installed apps
!adb shell pm list packages -3

# Take screenshot
!adb shell screencap -p /sdcard/screen.png && adb pull /sdcard/screen.png

# View logs
!adb logcat

# Push file
!adb push local_file.txt /sdcard/

# Pull file
!adb pull /sdcard/file.txt
```

### Troubleshooting:

**Emulator won't start?**
- Restart runtime: Runtime → Restart Runtime
- Make sure GPU is enabled

**Can't see emulator in VNC?**
- Wait 2-3 minutes for full boot
- Try refreshing the browser

**APK installation failed?**
- Check if device is online: `!adb devices`
- Try: `!adb kill-server && adb start-server`

---

## 📚 Resources:

- [Android ADB Documentation](https://developer.android.com/studio/command-line/adb)
- [Emulator Command Line](https://developer.android.com/studio/run/emulator-commandline)
- [Get Ngrok Auth Token](https://dashboard.ngrok.com/signup)

---

### 💡 Pro Tips:

1. **Save your work**: Download APKs/screenshots before session ends
2. **Get Ngrok token**: Free tier gives you better limits
3. **Use T4 GPU**: Better performance for emulator
4. **Session timeout**: Colab free tier = 12 hours max
5. **Keep tab open**: Browser must stay open for VNC access

---

## 🎉 You're All Set!

Your Android emulator is running in the cloud with:
- ✅ Custom phone screen size
- ✅ GPU acceleration
- ✅ Web-based remote access
- ✅ Full ADB control
- ✅ APK installation ready

**Enjoy testing your apps! 🚀**